# Importar ficheiros CRONO (cargas) para SQLite

1. Preparar BD (tabelas `cargas_2025` / `cargas_2026`)
2. Ler ficheiros CRONO da pasta
3. Extrair ano dos dados e inserir na BD correta
4. Se BD não existir: mostra as linhas mas não insere

In [8]:
import os
import csv
import glob
import platform
import sqlite3
from datetime import datetime
from pathlib import Path

import pandas as pd

if platform.system() == "Windows":
    DB_PATH = r"C:\Users\LISARR\Documents\python\00.DB\2026.db"
    PASTA_FICHEIROS = Path(r"C:\Users\LISARR\Desktop\02.Pontualidade\crono")
elif platform.system() == "Darwin":
    DB_PATH = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00.DB/2026.db"
    PASTA_FICHEIROS = Path("/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados")

print("DB_PATH:", DB_PATH)
print("PASTA_FICHEIROS:", PASTA_FICHEIROS)

DB_PATH: C:\Users\LISARR\Documents\python\00.DB\2026.db
PASTA_FICHEIROS: C:\Users\LISARR\Desktop\02.Pontualidade\crono


In [9]:
# 49 colunas, exatamente como aparecem no ficheiro
COLUNAS_CARGAS_ORDENADAS = [
    ("TIPO_SUMINISTRO", "Tipo Suministro"),
    ("BASE", "Base"),
    ("LANZADERA", "Lanzadera"),
    ("PUNTO_SUMINISTRO", "Punto Suministro"),
    ("RUTA", "Ruta"),
    ("PALES_TMS", "Palés TMS"),
    ("TEMPERATURA_REQUERIDA", "Temperatura"),
    ("AGENCIA", "Agencia"),
    ("TRANSPORTISTA", "Transportista"),
    ("DNI", "DNI"),
    ("PRECINTO", "Precinto"),
    ("TELEFONO", "Teléfono"),
    ("TRACTORA", "Tractora"),
    ("REMOLQUE", "Remolque"),
    ("FECHA_PREVISTA_POSICIONAMIENTO", "Fecha Prevista Posicionamiento"),
    ("HORA_PREVISTA_POSICIONAMIENTO", "Hora Prevista Posicionamiento"),
    ("FECHA_REAL_POSICIONAMIENTO", "Fecha Real Posicionamiento"),
    ("HORA_REAL_POSICIONAMIENTO", "Hora Real Posicionamiento"),
    ("FECHA_REAL_ENTRADA", "Fecha Real Entrada"),
    ("HORA_REAL_ENTRADA", "Hora Real Entrada"),
    ("MUELLE", "Muelle"),
    ("FECHA_PREVISTA_SALIDA", "Fecha Prevista Salida"),
    ("HORA_PREVISTA_SALIDA", "Hora Prevista Salida"),
    ("FECHA_REAL_SALIDA", "Fecha Real Salida"),
    ("HORA_REAL_SALIDA", "Hora Real Salida"),
    ("FECHA_PREVISTA_ENTREGA", "Fecha Prevista Entrega"),
    ("HORA_PREVISTA_ENTREGA", "Hora Prevista Entrega"),
    ("FECHA_REAL_ENTREGA", "Fecha Real Entrega"),
    ("HORA_REAL_ENTREGA", "Hora Real Entrega"),
    ("HORA_SALIDA_ENTREGA", "Hora Salida Entrega"),
    ("OBSERVACIONES", "Observaciones"),
    ("COMENTARIOS", "Comentarios"),
    ("ZONA", "Zona"),
    ("CLIENTE", "Cliente"),
    ("AUTORIZADO_AUTOCARGA", "Autorizado autocarga"),
    ("AUTOCARGA", "Autocarga"),
    ("HUECOS_TMS", "Huecos TMS"),
    ("HUECOS_CARGA", "Huecos carga"),
    ("HUECOS_DESCARGA", "Huecos descarga"),
    ("TEMPERATURA_MEDIDA1", "Temperatura"),
    ("TEMPERATURA_MEDIDA2", "Temperatura2"),
    ("MOTIVO", "Motivo"),
    ("ESTADO", "Estado"),
    ("ESTADO_MERCANCIA", "Estado mercancía"),
    ("ESTADO_CAJA", "Estado caja"),
    ("ESTADO_OLORES", "Estado olores"),
    ("ESTADO_LIMPIEZA_VEHICULO", "Estado limpieza vehículo"),
    ("ESTADO_VEHICULO_SECO", "Estado vehículo seco"),
    ("ESTADO_LIBRE_PLAGAS", "Estado libre de plagas"),
]

COLUNAS_CARGAS = {nome_sql: nome_ficheiro for nome_sql, nome_ficheiro in COLUNAS_CARGAS_ORDENADAS}
ORDEM_CABECALHO_CARGAS = [nome_ficheiro for _, nome_ficheiro in COLUNAS_CARGAS_ORDENADAS]

COLUNA_ORIGEM = "ficheiro_origem"
ANOS = ("2025", "2026")
CHAVE_UNICA_CARGAS = tuple(COLUNAS_CARGAS.keys())

print(f"Colunas definidas: {len(COLUNAS_CARGAS)}")

Colunas definidas: 49


In [10]:
def preparar_base_dados(con):
    """Cria tabelas cargas_<ano> com indice unico."""
    colunas_sql = ",\n        ".join(f'"{c}" TEXT' for c in COLUNAS_CARGAS.keys())
    chave_sql = ", ".join(f'COALESCE("{c}", \'\')' for c in CHAVE_UNICA_CARGAS)

    cur = con.cursor()
    for ano in ANOS:
        cur.execute(f'''
            CREATE TABLE IF NOT EXISTS cargas_{ano} (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                {colunas_sql},
                "{COLUNA_ORIGEM}" TEXT
            )
        ''')
        cur.execute(f'DROP INDEX IF EXISTS idx_cargas_{ano}_chave')
        cur.execute(
            f'CREATE UNIQUE INDEX idx_cargas_{ano}_chave '
            f'ON cargas_{ano}({chave_sql})'
        )
    con.commit()

pasta_db = os.path.dirname(DB_PATH)
if pasta_db and not os.path.exists(pasta_db):
    os.makedirs(pasta_db, exist_ok=True)

con = sqlite3.connect(DB_PATH)
preparar_base_dados(con)
con.close()
print("Base de dados, tabelas e índices prontos.")

Base de dados, tabelas e índices prontos.


In [11]:
def listar_ficheiros_excel(pasta):
    """Procura ficheiros .xls (também em subpastas)."""
    encontrados = glob.glob(os.path.join(pasta, "**", "*.xls"), recursive=True)
    vistos = set()
    ficheiros = []
    for caminho in encontrados:
        chave = os.path.normcase(os.path.abspath(caminho))
        if chave not in vistos:
            vistos.add(chave)
            ficheiros.append(caminho)
    return sorted(ficheiros)

def validar_data(valor):
    """Valida uma data DD/MM/AAAA."""
    if not valor:
        return None
    texto = str(valor).strip()
    if not texto or texto.upper() == "N/D":
        return None
    try:
        return datetime.strptime(texto, "%d/%m/%Y")
    except ValueError:
        return None

def ler_linhas_csv_cargas(caminho_ficheiro):
    """Lê um ficheiro CRONO (tab-separated, ISO-8859-1)."""
    with open(caminho_ficheiro, encoding="iso-8859-1", newline="") as f:
        linhas_ficheiro = list(csv.reader(f, delimiter="\t"))

    if not linhas_ficheiro:
        return [], []

    cabecalho = linhas_ficheiro[0]
    linhas = linhas_ficheiro[1:]

    n_colunas = len(cabecalho)
    linhas_normalizadas = []
    for valores in linhas:
        if len(valores) < n_colunas:
            valores = valores + [None] * (n_colunas - len(valores))
        elif len(valores) > n_colunas:
            valores = valores[:n_colunas]
        linhas_normalizadas.append(valores)

    return cabecalho, linhas_normalizadas

ficheiros = listar_ficheiros_excel(PASTA_FICHEIROS)
print(f"Ficheiros encontrados: {len(ficheiros)}")

Ficheiros encontrados: 3


In [12]:
def extrair_ano_e_data_cargas(valores):
    """
    Extrai ano e data de uma linha.
    Procura FECHA_PREVISTA_POSICIONAMIENTO (índice 14).
    Devolve (ano_str, data_str) ou (None, None).
    """
    colunas_ordem = list(COLUNAS_CARGAS.keys())
    try:
        idx_fecha = colunas_ordem.index("FECHA_PREVISTA_POSICIONAMIENTO")
        if idx_fecha < len(valores):
            data_str = valores[idx_fecha]
            data_obj = validar_data(data_str)
            if data_obj:
                ano_str = str(data_obj.year)
                return ano_str, data_str
    except (ValueError, IndexError):
        pass
    return None, None

def montar_linha_cargas(valores, nome_ficheiro):
    """
    Monta linha pronta para INSERT.
    Devolve tupla (valor1, valor2, ..., ficheiro_origem).
    """
    colunas_ordem = list(COLUNAS_CARGAS.keys())
    linha = tuple(valores[:len(colunas_ordem)]) + (nome_ficheiro,)
    return linha

# Preparar statements SQL de INSERT
sql_insercao = {}
for ano in ANOS:
    colunas = ",".join(f'"{c}"' for c in COLUNAS_CARGAS.keys())
    placeholders = ",".join(["?"] * (len(COLUNAS_CARGAS) + 1))
    sql_insercao[ano] = f'INSERT INTO cargas_{ano} ({colunas}, "{COLUNA_ORIGEM}") VALUES ({placeholders})'

print("Funções e statements prontos.")

Funções e statements prontos.


In [13]:
# ==========================
# IMPORTAR COM VERIFICAÇÃO DE BD
# ==========================

contagem_novas = {ano: 0 for ano in ANOS}
total_duplicadas = 0
total_sem_data = 0
total_sem_bd = 0
total_processados = 0
total_cabecalho_invalido = 0
total_erros = 0

linhas_sem_bd = []
linhas_com_erro = []

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

# Verificar quais as BDs existentes
cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'cargas_%'")
tabelas_existentes = {row[0] for row in cur.fetchall()}
anos_disponiveis = {int(t.split('_')[1]) for t in tabelas_existentes if t.startswith('cargas_')}
anos_disponiveis_str = {str(a) for a in anos_disponiveis}

print(f"Bases disponíveis para anos: {sorted(anos_disponiveis_str)}\n")

# Processar ficheiros
for i, caminho in enumerate(ficheiros, start=1):
    nome_ficheiro = os.path.basename(caminho)

    try:
        cabecalho, linhas = ler_linhas_csv_cargas(caminho)

        if cabecalho != ORDEM_CABECALHO_CARGAS:
            total_cabecalho_invalido += 1
            print(
                f"[{i}/{len(ficheiros)}] [AVISO] "
                f"cabecalho de '{nome_ficheiro}' diferente - ficheiro ignorado."
            )
            continue

        novas_ficheiro = {ano: 0 for ano in ANOS}
        duplicadas_ficheiro = 0
        sem_data_ficheiro = 0
        sem_bd_ficheiro = 0
        erro_ficheiro = 0

        for idx_linha, valores in enumerate(linhas):
            if all(v is None or str(v).strip() == "" for v in valores):
                continue

            ano, _ = extrair_ano_e_data_cargas(valores)

            if ano not in ANOS:
                sem_data_ficheiro += 1
                continue

            # Verificar se a BD para este ano existe
            if ano not in anos_disponiveis_str:
                sem_bd_ficheiro += 1
                total_sem_bd += 1
                linha = montar_linha_cargas(valores, nome_ficheiro)
                linhas_sem_bd.append((ano, linha, idx_linha + 2))  # +2: +1 cabeçalho, +1 índice
                continue

            # Inserir normalmente
            try:
                linha = montar_linha_cargas(valores, nome_ficheiro)
                cur.execute(sql_insercao[ano], linha)

                if cur.rowcount == 1:
                    novas_ficheiro[ano] += 1
                else:
                    duplicadas_ficheiro += 1
            except sqlite3.IntegrityError:
                # Duplicado dentro do ficheiro
                duplicadas_ficheiro += 1
            except Exception as e:
                erro_ficheiro += 1
                linhas_com_erro.append((ano, str(e), idx_linha + 2))

        # Confirma este ficheiro
        con.commit()
        total_processados += 1

        for ano in ANOS:
            contagem_novas[ano] += novas_ficheiro[ano]

        total_duplicadas += duplicadas_ficheiro
        total_sem_data += sem_data_ficheiro

        resumo = f"[{i}/{len(ficheiros)}] {nome_ficheiro} -> "
        partes = []
        if novas_ficheiro['2025'] > 0:
            partes.append(f"{novas_ficheiro['2025']} novas 2025")
        if novas_ficheiro['2026'] > 0:
            partes.append(f"{novas_ficheiro['2026']} novas 2026")
        if duplicadas_ficheiro > 0:
            partes.append(f"{duplicadas_ficheiro} dup")
        if sem_bd_ficheiro > 0:
            partes.append(f"[SEM BD] {sem_bd_ficheiro}")
        if sem_data_ficheiro > 0:
            partes.append(f"{sem_data_ficheiro} sem data")
        if erro_ficheiro > 0:
            partes.append(f"[ERRO] {erro_ficheiro}")
        
        print(resumo + (" | ".join(partes) if partes else "nenhuma"))

    except Exception as erro:
        con.rollback()
        total_erros += 1
        print(f"[{i}/{len(ficheiros)}] [ERRO] '{nome_ficheiro}': {erro}")

con.close()

print("\n" + "="*70)
print(f"2025: {contagem_novas['2025']} novas | 2026: {contagem_novas['2026']} novas | Dup: {total_duplicadas} | Sem BD: {total_sem_bd}")
print("="*70)

Bases disponíveis para anos: ['2025', '2026']

[1/3] crono_1787930508519.xls -> 284 novas 2026 | 1704 dup
[2/3] crono_1787930535978.xls -> 1963 dup
[3/3] crono_1787930553151.xls -> 462 dup

2025: 0 novas | 2026: 284 novas | Dup: 4129 | Sem BD: 0


In [14]:
# === MOSTRAR LINHAS SEM BD ===

if linhas_sem_bd:
    print(f"\n{'='*70}")
    print(f"LINHAS SEM BASE DE DADOS ({len(linhas_sem_bd)} total):")
    print(f"{'='*70}\n")
    
    colunas_ordem = list(COLUNAS_CARGAS.keys())
    dados = []
    for ano, linha, num_linha in linhas_sem_bd:
        dados.append({
            'ANO': ano,
            'LINHA_FICHEIRO': num_linha,
            **{col: linha[i] if i < len(linha) else None 
               for i, col in enumerate(colunas_ordem)}
        })
    
    df_sem_bd = pd.DataFrame(dados)
    
    for ano in sorted(df_sem_bd['ANO'].unique()):
        df_ano = df_sem_bd[df_sem_bd['ANO'] == ano].drop('ANO', axis=1)
        print(f"\nAno {ano} ({len(df_ano)} linhas):")
        df_ano.head(100)
else:
    print("\n✓ Todas as linhas foram processadas com sucesso (nenhuma sem BD).")

if linhas_com_erro:
    print(f"\n{'='*70}")
    print(f"LINHAS COM ERRO NA INSERÇÃO ({len(linhas_com_erro)}):")
    print(f"{'='*70}")
    for ano, erro, num_linha in linhas_com_erro[:10]:
        print(f"  Ano {ano}, linha {num_linha}: {erro}")


✓ Todas as linhas foram processadas com sucesso (nenhuma sem BD).
